# Automatic Vectorization with vmap

This notebook explores JAX's `vmap` transformation, which automatically vectorizes functions to operate over batches of data without writing explicit loops.

## Why vmap?

### 🧠 The Core Concept

**Traditional approach:**
```python
results = [f(x) for x in batch]  # Python loop — slow
```

**JAX approach:**
```python
batched_f = jax.vmap(f)
results = batched_f(batch)  # Single vectorized call — fast
```

**Why this matters:**
- **No Python loops** — hardware does the batching
- **No manual reshaping** — vmap handles dimensions
- **Composable** — combine with `jit` and `grad` freely
- **Write once** — function for one example, vmap for a batch

In [1]:
# Import JAX libraries
import jax
import jax.numpy as jnp

### ✅ Exercise 51 — vmap a Simple Function

Write a function for a single element, then use `vmap` to apply it across a batch. No loops needed.

In [2]:
# Function that works on a single scalar
def square_plus_one(x):
    return x ** 2 + 1

# vmap lifts it to work on a batch
batch_fn = jax.vmap(square_plus_one)

xs = jnp.array([1.0, 2.0, 3.0, 4.0])
batch_fn(xs)  # → [2., 5., 10., 17.]

Array([ 2.,  5., 10., 17.], dtype=float32)

### ✅ Exercise 52 — vmap vs Python Loop

Compare the performance of `vmap` against a naive Python loop. vmap pushes the loop into XLA where it can be parallelized.

In [3]:
import time

def single_fn(x):
    return jnp.sin(x) * jnp.cos(x)

batch = jnp.linspace(0, 10, 10000)

# Python loop approach
def loop_apply(batch):
    return jnp.stack([single_fn(x) for x in batch])

# vmap approach
vmap_apply = jax.jit(jax.vmap(single_fn))

# Warm up
vmap_apply(batch)

def benchmark(fn, x, runs=50):
    start = time.perf_counter()
    for _ in range(runs):
        jax.block_until_ready(fn(x))
    return (time.perf_counter() - start) / runs

vmap_time = benchmark(vmap_apply, batch)
loop_time = benchmark(loop_apply, batch)

print(f"vmap + JIT : {vmap_time * 1e3:.2f} ms")
print(f"Python loop: {loop_time * 1e3:.2f} ms")
print(f"Speedup    : {loop_time / vmap_time:.1f}x")

vmap + JIT : 0.06 ms
Python loop: 367.81 ms
Speedup    : 6309.6x


### Understanding in_axes Step by Step

`in_axes` tells vmap which axis of each argument contains the batch dimension:

- `0` = batch over axis 0 (rows for first argument)
- `None` = don't batch, broadcast this same value

**What happens with `in_axes=(0, None)`:**

```
Before vmap:
batch_vecs = [[1, 2],   ← batch item 0
              [3, 4],   ← batch item 1  
              [5, 6]]   ← batch item 2
weights = [0.5, -0.5]   ← same for all batch items

After vmap with in_axes=(0, None):
Call 1: dot_with([1, 2], [0.5, -0.5]) = -0.5
Call 2: dot_with([3, 4], [0.5, -0.5]) = -0.5
Call 3: dot_with([5, 6], [0.5, -0.5]) = -0.5

Result: [-0.5, -0.5, -0.5]
```

This pattern is common in neural networks: batch inputs with shared weights.

In [4]:
# Dot product of each row in a batch with a single vector
def dot_with(vec, w):
    return jnp.dot(vec, w)

batch_vecs = jnp.array([[1.0, 2.0],
                        [3.0, 4.0],
                        [5.0, 6.0]])  # (3, 2) — 3 vectors

weights = jnp.array([0.5, -0.5])     # (2,) — shared across batch

# in_axes=(0, None): batch over first arg axis 0, broadcast second arg (don't batch)
batched_dot = jax.vmap(dot_with, in_axes=(0, None))
batched_dot(batch_vecs, weights)  # → [-0.5, -0.5, -0.5]

Array([-0.5, -0.5, -0.5], dtype=float32)

### ✅ Exercise 54 — vmap Over Columns (in_axes=1)

By changing `in_axes`, you can vectorize over any dimension — not just the first.

In [5]:
# Sum each column of a matrix
matrix = jnp.array([[1.0, 2.0, 3.0],
                    [4.0, 5.0, 6.0]])  # (2, 3)

# vmap over axis 1 (columns) — each slice is a column vector
col_sum = jax.vmap(jnp.sum, in_axes=1)
col_sum(matrix)  # → [5., 7., 9.]

Array([5., 7., 9.], dtype=float32)

### ✅ Exercise 55 — out_axes Control

`out_axes` controls where the batch dimension appears in the output. Default is 0 (first axis).

### The Mental Model

- **`in_axes`**: "Where to find the batch dimension in inputs"
- **`out_axes`**: "Where to put the batch dimension in outputs"

In [ ]:
def scale(x):
    return x * 10.0

xs = jnp.array([[1.0, 2.0],
                [3.0, 4.0],
                [5.0, 6.0]])  # (3, 2)

# out_axes=0: batch dim is first (default) → shape (3, 2)
result_0 = jax.vmap(scale, out_axes=0)(xs)
print(f"out_axes=0: shape {result_0.shape}")
print(result_0)

# out_axes=1: batch dim is second → shape (2, 3)
# It's like transposing the output
result_1 = jax.vmap(scale, out_axes=1)(xs)
print(f"\nout_axes=1: shape {result_1.shape}")
print(result_1)

out_axes=0: shape (3, 2)
[[10. 20.]
 [30. 40.]
 [50. 60.]]

out_axes=1: shape (2, 3)
[[10. 30. 50.]
 [20. 40. 60.]]


### ✅ Exercise 56 — Batched Matrix-Vector Multiply

A common deep learning pattern: apply the same weight matrix to a batch of input vectors. This is the core of a linear layer.

In [ ]:
# Linear layer: y = Wx for each x in a batch
def linear(W, x):
    return W @ x

key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)

W = jax.random.normal(k1, (4, 3))       # (4, 3) weight matrix
X_batch = jax.random.normal(k2, (8, 3))  # 8 samples, 3 features each

# Batch over x (axis 0), broadcast W (None)
batched_linear = jax.vmap(linear, in_axes=(None, 0))
Y = batched_linear(W, X_batch)  # (8, 4)
print(f"Input batch:  {X_batch.shape}")
print(f"Weight:       {W.shape}")
print(f"Output batch: {Y.shape}")

Input batch:  (8, 3)
Weight:       (4, 3)
Output batch: (8, 4)


W0304 13:48:43.894613 10414995 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


### ✅ Exercise 57 — Nested vmap (Double Batching)

vmap can be nested to handle multiple batch dimensions — for example, batching over both inputs in a pairwise computation.

### Manual Calculation

**For a1 = [1, 0]:**
- dist([1,0], [1,1]) = (1-1)² + (0-1)² = 0 + 1 = 1
- dist([1,0], [0,0]) = (1-0)² + (0-0)² = 1 + 0 = 1
- dist([1,0], [2,2]) = (1-2)² + (0-2)² = 1 + 4 = 5

**For a2 = [0, 1]:**
- dist([0,1], [1,1]) = (0-1)² + (1-1)² = 1 + 0 = 1
- dist([0,1], [0,0]) = (0-0)² + (1-0)² = 0 + 1 = 1
- dist([0,1], [2,2]) = (0-2)² + (1-2)² = 4 + 1 = 5

In [8]:
# Compute pairwise squared distances between two sets of vectors
def squared_dist(a, b):
    return jnp.sum((a - b) ** 2)

A = jnp.array([[1.0, 0.0],
               [0.0, 1.0]])  # 2 vectors

B = jnp.array([[1.0, 1.0],
               [0.0, 0.0],
               [2.0, 2.0]])  # 3 vectors

# Outer vmap: for each a in A
#   Inner vmap: compute distance to every b in B
pairwise = jax.vmap(jax.vmap(squared_dist, in_axes=(None, 0)), in_axes=(0, None))
distances = pairwise(A, B)  # (2, 3) distance matrix
print(f"Shape: {distances.shape}")
print(distances)

Shape: (2, 3)
[[1. 1. 5.]
 [1. 1. 5.]]


### ✅ Exercise 58 — vmap + grad (Per-Sample Gradients)

Combine `vmap` and `grad` to compute per-sample gradients — something that is difficult in most frameworks but trivial in JAX.

In [9]:
# Per-sample gradient: gradient of loss w.r.t. weights for EACH sample
def loss_fn(w, x):
    return jnp.sum((w * x) ** 2)

w = jnp.array([1.0, 2.0, 3.0])
X = jnp.array([[1.0, 0.0, 0.0],
               [0.0, 1.0, 0.0],
               [0.0, 0.0, 1.0],
               [1.0, 1.0, 1.0]])

# grad w.r.t. w (argnums=0), then vmap over x (in_axes=(None, 0))
per_sample_grad = jax.vmap(jax.grad(loss_fn, argnums=0), in_axes=(None, 0))
grads = per_sample_grad(w, X)  # (4, 3) — one gradient vector per sample
print(f"Per-sample gradients shape: {grads.shape}")
print(grads)

Per-sample gradients shape: (4, 3)
[[2. 0. 0.]
 [0. 4. 0.]
 [0. 0. 6.]
 [2. 4. 6.]]


### ✅ Exercise 59 — vmap + jit (Full Composition)

The real power of JAX: compose `vmap`, `grad`, and `jit` together for a compiled, batched gradient computation.

In [10]:
import time

def model(w, x):
    return jnp.sum(jnp.tanh(w * x))

# Compose: jit(vmap(grad))
fast_per_sample_grad = jax.jit(jax.vmap(jax.grad(model), in_axes=(None, 0)))

key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)
w = jax.random.normal(k1, (16,))
X = jax.random.normal(k2, (1024, 16))

# Warm up
fast_per_sample_grad(w, X)

def benchmark(fn, *args, runs=200):
    start = time.perf_counter()
    for _ in range(runs):
        jax.block_until_ready(fn(*args))
    return (time.perf_counter() - start) / runs

eager_fn = jax.vmap(jax.grad(model), in_axes=(None, 0))
eager_fn(w, X)  # warm up

jit_time   = benchmark(fast_per_sample_grad, w, X)
eager_time = benchmark(eager_fn, w, X)

print(f"jit(vmap(grad)): {jit_time * 1e3:.2f} ms")
print(f"vmap(grad)     : {eager_time * 1e3:.2f} ms")
print(f"Speedup        : {eager_time / jit_time:.1f}x")

jit(vmap(grad)): 0.04 ms
vmap(grad)     : 1.70 ms
Speedup        : 41.8x


### ✅ Exercise 60 — vmap a Custom Neural Network Layer

Put it all together: define a single-sample forward pass, then vmap it into a batched layer with per-sample gradients.

### Traditional vs JAX Approach

**Traditional approach (PyTorch/TensorFlow):**
```python
# Manual loop over batch
grads = []
for x, y in batch:
    grad = compute_grad(params, x, y)
    grads.append(grad)
```

**JAX approach:**
```python
# One line, no Python loops
batched_grad = jax.jit(jax.vmap(jax.grad(forward), in_axes=(None, 0)))
```

In [ ]:
# Single-sample forward pass
def forward(params, x):
    W, b = params
    return jnp.sum(jnp.tanh(W @ x + b)) # For gradients to work, the function must return a scalar

# Initialize parameters
key = jax.random.PRNGKey(7)
k1, k2 = jax.random.split(key)
W = jax.random.normal(k1, (4, 3))   # 4 outputs, 3 inputs
b = jax.random.normal(k2, (4,))     # 4 biases
params = (W, b)

# Single input
x = jnp.array([1.0, 2.0, 3.0])
print(f"Single forward: {forward(params, x):.4f}")

# Batch of inputs
X_batch = jax.random.normal(jax.random.PRNGKey(99), (8, 3))

# Batched forward: vmap over x, broadcast params
batched_forward = jax.vmap(forward, in_axes=(None, 0))
print(f"Batch forward shape: {batched_forward(params, X_batch).shape}")

# Batched gradient w.r.t. params for each sample
batched_grad = jax.jit(jax.vmap(jax.grad(forward), in_axes=(None, 0)))
grads = batched_grad(params, X_batch)
dW, db = grads
print(f"Per-sample dW shape: {dW.shape}")  # (8, 4, 3)
print(f"Per-sample db shape: {db.shape}")  # (8, 4)

Single forward: 1.0221
Batch forward shape: (8,)
Per-sample dW shape: (8, 4, 3)
Per-sample db shape: (8, 4)


## Conclusion

### The vmap Philosophy

**Write for one example. vmap for a batch.**

| Pattern | What it does |
|---|---|
| `vmap(f)` | Batch a function over its first argument |
| `vmap(f, in_axes=(0, None))` | Batch first arg, broadcast second |
| `vmap(f, in_axes=1)` | Batch over columns instead of rows |
| `vmap(vmap(f))` | Double batch (e.g., pairwise ops) |
| `jit(vmap(grad(f)))` | Compiled batched gradients |

This composability is what makes JAX unique:
- **grad** — automatic differentiation
- **vmap** — automatic vectorization
- **jit** — automatic compilation

All three are **function transformations** that compose freely, enabling you to write simple single-example code and scale it to production workloads.